In [7]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
#from scrapy import fetch_website_links, fetch_website_contents
from openai import OpenAI
import ollama
from IPython.display import Markdown, display

from parsing_url import redacted_links, redacted_contents, \
    redacted_images, redacted_headings, redacted_metadata


#### How multiple model compleated their chat

1. client.chat.completions.create
    - Library: OpenAI Python client (classic API).
    - Purpose: Create a chat completion (like ChatGPT).
    - Usage: You send a list of messages (system, user, assistant) and get the model’s response.
    - Best for: Conversational AI, multi-turn chat, structured prompts.
    - Example:

`response = client.chat.completions.create( model="gpt-4o", messages=[{"role": "user", "content": "Explain transformers"}])`

2. client.models.generate_content:
    - Library: Google Gemini client (Generative Language API).
    - Purpose: Generate text or multimodal content (text, images, code).
    - Usage: You call the model directly with input text or other modalities.
    - Best for: Multimodal tasks (Gemini supports text + images), single-shot generation.
    - Example

`response = client.models.generate_content(model="gemini-1.5-pro",contents="Explain transformers in simple terms")`

3. client.messages.create
    - Library: Anthropic Python client (Claude API).
    - Purpose: Create a message completion (Claude’s equivalent of chat).
    - Usage: You send a conversation history and get Claude’s reply.
    - Best for: Claude’s conversational interface, agentic workflows.
    - Example:

    `response = client.messages.create(model="claude-3-opus",messages=[{"role": "user", "content": "Explain transformers"}])`



#### What is prompt caching ?

Prompt caching is a technique used in modern LLM APIs (like OpenAI, Claude, Gemini) to reduce cost and latency by reusing the unchanged prefix of a prompt across multiple requests. Instead of recomputing the same long instructions or context every time, the model caches its internal state for that prefix and only processes the new, variable part of the input.

- **Purpose**: Speeds up API responses and lowers costs by reusing cached prompt prefixes instead of recomputing them.
- **Automatic Behavior**: Enabled by default for eligible requests (≥1024 tokens).

#### For further understanding first we need to understand What is a Token?

A token is a chunk of text the model reads.

Roughly:

`1 token ≈ 4 characters of English text`

`1 token ≈ ¾ of a word`

`Example:`

`"Hello world!" → 3 tokens`

`"You are a helpful assistant." → 6 tokens`

So when we say 1500 tokens, it means the prompt is long — about 1,000–1,200 words of text.

#### Prompt Caching Flow:

![Prompt Caching](image.png)         

Cache Hit vs Cache Miss in the context of OpenAI’s prompt caching:

#### Cache Hit:

A cache hit happens when the prefix of your prompt (the first 1024+ tokens, or the part marked with a cache key/breakpoint) matches exactly with something already stored in the cache.

#### Cache Miss:

A cache miss happens when the prefix is different from what’s stored in cache, so the model must recompute it.

`Cache Hit = Prefix is identical, stable, and reused.`

`Cache Miss = Prefix is new, altered, or expired.`

#### Example: Cache Hit

`Prompt 1 (first request):`

System: You are a helpful assistant. Here are 1500 tokens of detailed instructions, examples, and schemas for how to answer user questions about math problems...

User: Solve 2x + 5 = 15

`Prompt 2 (second request):`

System: You are a helpful assistant. Here are the same 1500 tokens of detailed instructions, examples, and schemas for how to answer user questions about math problems...

User: Solve 3x - 7 = 11

The prefix (system + examples) is identical and ≥1024 tokens.Only the user query changes at the end.
Result: Cache Hit → the 1500‑token prefix is reused from cache, saving cost and latency.

#### Example: Cache Miss:

Prompt 1 (first request):

System: You are a helpful assistant. Here are 1500 tokens of detailed instructions, examples, and schemas for how to answer user questions about math problems...

User: Solve 2x + 5 = 15

`Prompt 2 (second request, slightly modified):`

System: You are a helpful assistant. Here are 1500 tokens of detailed instructions, examples, and schemas for how to answer user questions about physics problems...

User: Solve F = ma for m

The prefix changed (“math problems” → “physics problems”). Even though most of the text is the same, the cache sees it as different. Result: Cache Miss → full recomputation, no reuse.

#### Q. Implicit vs explicit variable in programming language ?

Implicit variables are declared automatically by the system when you first use them, while explicit variables require you to manually write a specific declaration statement before usage.

`int age = 26;                      // Explicitly an integer`

`double salary = 85000.50;          // Explicitly a double-precision float`

`std::string name = "Alice";        // Explicitly a string object`

`auto count = 10;                   // Implicitly  Deduced as int`

`auto price = 19.99;                // Implicitly Deduced as double`

#### Implicit Prompt Caching:

The default mode. The API automatically places a cache breakpoint at the latest message in your prompt.
Everything before that breakpoint (system instructions, examples, schemas) is cached if it’s ≥1024 tokens. You don’t need to manually mark anything — caching just happens.

#### Example (implicit mode):
The system + instructions are cached automatically. Only the user’s query changes → Cache Hit.

In [ ]:
{
  "messages": [
    {"role": "system", "content": "You are a math tutor. [1200 tokens of instructions]"},
    {"role": "user", "content": "Solve 2x + 5 = 15"}
  ],
  "prompt_cache_breakpoint": "implicit"
}

#### Explicit Prompt Caching:

Definition: You manually control where caching happens. You insert explicit breakpoints in your prompt to mark reusable segments. Only those marked segments are cached; nothing else is cached automatically.
Useful when you want fine‑grained control (e.g., multiple reusable blocks).

In [ ]:
{
  "messages": [
    {"role": "system", "content": "You are a math tutor. [1200 tokens of instructions]", 
     "prompt_cache_breakpoint": "explicit"},
    {"role": "user", "content": "Solve 2x + 5 = 15"}
  ]
}


The system + instructions are cached because you explicitly marked them. If you don’t mark a section, it won’t be cached. Gives you control to decide which parts are stable vs dynamic.

#### Training vs Inference time scaling

Training-time scaling means increasing model size, data, and compute during pretraining to improve general capability, while inference-time scaling means spending more compute at deployment (test-time) — e.g., generating more tokens, running multiple reasoning paths, or using search strategies — to boost performance without retraining.

#### Training-Time Scaling:

Expanding compute during the pretraining phase.
- Methods:
    - Increase model parameters (e.g., 7B → 70B).
    - Train on more tokens/data.
    - Use longer training runs with larger batch sizes.
    - Goal: Build a stronger base model with higher general accuracy.
    - Cost Profile: One-time, massive expense (GPU/TPU clusters, weeks of training).
    - Scaling Laws: Kaplan (2020) and Chinchilla (2022) showed optimal trade-offs between model size and dataset size.
    - Risks & Trade-offs: Expensive, limited by data availability.

#### Inference-Time Scaling

Allocating extra compute after training, during deployment/test-time. Improve reasoning quality without retraining. Variable, per-query expense (more tokens per answer). Inference scaling: Risk of overthinking — accuracy can drop if reasoning chains are too long (inverted U-curve performance)
- Methods:
    - Chain-of-thought (CoT) reasoning → let the model “think longer.”
    - Best-of-N sampling → generate multiple answers, pick the best.
    - Tree search / beam search → explore reasoning paths.
    - Majority voting / ensembles → combine outputs for accuracy.

Mathmetical Example 

TODO

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [3]:
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [4]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))
response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))
response = openai.chat.completions.create(model="gpt-5-mini", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

1/2

2/3

2/3

#### reasoning_effort parameter:

how much compute to spend “thinking” before answering.
- Minimal → The model gives a quick, low‑compute answer.
- Low / Medium / High → The model spends more tokens internally, exploring reasoning chains, before producing the final output.
- Effect: Higher effort usually improves accuracy on reasoning tasks (math puzzles, logic, planning), but costs more tokens and latency.

 In Ollama there isn’t a completions.create method like in the OpenAI client. Ollama’s Python API is simpler: chat completions happen through ollama.chat(), not through a nested .completions object.

In [11]:
response = ollama.chat(model="llama3.2", messages=easy_puzzle)
display(Markdown(response['message']['content']))
#response = ollama.chat(model="gpt-oss:20b", messages=easy_puzzle)
#display(Markdown(response['message']['content']))

0.5

The Anthropic Client Library is the official Python SDK for interacting with Anthropic’s models (Claude family). It provides a structured way to send prompts, manage conversations, and receive responses from Anthropic’s API — similar to how OpenAI’s openai library works for GPT models.

#### Q. What is lightweight LiteLLM?

LiteLLM is a lightweight Python library that acts as a unified interface for calling multiple LLM providers (OpenAI, Anthropic, Ollama, Hugging Face, Azure, etc.) with the same API style. Think of it as a “compatibility layer” — you write one piece of code, and you can swap the backend model provider without rewriting everything.

- Provides a single API (litellm.completion(), litellm.chat_completion()) that looks like OpenAI’s client.
- Supports many backends: OpenAI GPT, Anthropic Claude, Ollama local models, Hugging Face, Azure OpenAI, etc.
- Handles rate limiting, retries, logging, and caching automatically.
- Lets you switch providers easily by changing the model string (e.g., "gpt-4", "claude-3", "ollama/llama2").

#### Why It’s Different from OpenAI’s Client

OpenAI Client Scope Only OpenAI models while LiteLLM Scope Multiple providers (OpenAI, Anthropic, Ollama, HF, Azure, etc.)

👉 In short: LiteLLM = one lightweight client to rule them all.  

- LiteLLM: 
    - Avoid vendor lock‑in → same code works with OpenAI, Anthropic, Ollama, etc.
    - Cost optimization → route requests to cheaper providers depending on workload.
    - Resilience → fallback to another provider if one API is down.
    - Experimentation → benchmark different models without rewriting code.
    - Hybrid deployments → mix local models (Ollama) with cloud models (OpenAI/Anthropic).

Example of uses

`response = litellm.completion(model='', messages='')`

`print(response.choices[0].message.content)`